# EDA: NSL-KDD before any modeling

The point of this notebook is to understand the data before training anything.
Questions I want answered:

1. How imbalanced is the class problem?
2. Which features are categorical, which are numeric?
3. What do the attack families look like?
4. Any obvious red flags (leakage, zero-variance columns, garbage values)?


In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make the src/ package importable whether you run from notebooks/ or repo root
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.abspath(os.pardir))
else:
    sys.path.insert(0, os.getcwd())

from src.data.load import load_data
from src.data.preprocess import family_label as family

train = load_data(test=False)
test = load_data(test=True)
print(f'train: {train.shape}  test: {test.shape}')

## 1. Class balance (normal vs attack)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, df, name in zip(axes, [train, test], ['train', 'test']):
    vc = (df['label'] == 'normal').value_counts().sort_index()
    vc.index = ['attack', 'normal']
    vc.plot(kind='bar', ax=ax, color=['coral', 'steelblue'])
    ax.set_title(f'{name} - normal vs attack')
    ax.set_ylabel('count')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print('train attack ratio:', round((train['label'] != 'normal').mean(), 4))
print('test  attack ratio:', round((test['label'] != 'normal').mean(), 4))

## 2. Attack families

In [ ]:
for name, df in [('train', train), ('test', test)]:
    print(f'--- {name} ---')
    print(family(df['label']).value_counts().to_string())
    print()

## 3. Categorical features

protocol_type, service and flag are the only object columns. One-hot encoding
these will explode the feature count - service alone has ~70 values.

In [ ]:
cats = ['protocol_type', 'service', 'flag']
for c in cats:
    print(f'{c}: {train[c].nunique()} unique -> {sorted(train[c].unique())[:8]}...')
print('\nna counts:', train.isna().sum().sum())
print('total rows:', len(train))

## 4. Quick look at numeric ranges

These are wild. src_bytes goes up to millions while error rates sit in [0, 1].
Standardization is mandatory before KNN / LR.

In [ ]:
num_cols = [c for c in train.columns if c not in cats + ['label', 'difficulty']]
train[num_cols].describe().T[['min', '50%', 'max']].head(10)

## 5. Zero-variance / leakage check

In [ ]:
# Columns that never change won't help a classifier
zero_var = [c for c in num_cols if train[c].nunique() <= 1]
print('zero-variance columns:', zero_var)

# num_outbound_cmds is famously always 0 in NSL-KDD
print('\nnum_outbound_cmds unique values:', train['num_outbound_cmds'].unique())

## Takeaways

- Train is ~46.5% attack, test is ~43%. Not wildly imbalanced at the binary
  level, but the **families** are: R2L and U2R are tiny in train.
- Standardize everything numeric. One-hot the 3 categoricals.
- Drop zero-variance columns (num_outbound_cmds) or let the model ignore them.
- Accuracy will look fine because of the majority class - use F1 / ROC-AUC.

Next step: multi-class family classification and a zero-day style eval where
the model only ever saw some attack families during training.